### 1. Import Libraries & Dataset

In [4]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from copy import deepcopy


from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings("ignore")

### 2. Laoding the Data

In [6]:

df = pd.read_csv('./kaggle_dataset/data.csv')
df.head()

,Date,Open,High,Low,Close
0,2000-01-03,1482.15,1592.90,1482.15,1592.2
1,2000-01-04,1594.40,1641.95,1594.40,1638.7
2,2000-01-05,1634.55,1635.50,1555.05,1595.8
3,2000-01-06,1595.80,1639.00,1595.80,1617.6
4,2000-01-07,1616.60,1628.25,1597.20,1613.3


### 3. Splitting the Data

In [23]:
def return_pairs(column, days):
    pricess = list(column)
    X = []
    y = []
    for i in range(len(pricess) - days):
        X.append(pricess[i:i+days])
        y.append(pricess[i+days])
    return np.array(X), np.array(y)

target_columns =  ['High']
day_chunks =  [30, 60, 90]

chunked_data = {}

for col in target_columns:
    for days in day_chunks:
        key_X = f"X_{col}_{days}"
        key_y = f"y_{col}_{days}"
        X, y = return_pairs(df[col], days)
        chunked_data[key_X] = X
        chunked_data[key_y] = y


chunk_pairs = []

for key in chunked_data.keys():
    if key.startswith("X_"):
        y_key = key.replace("X_", "y_")
        if y_key in chunked_data:
            chunk_pairs.append([key, y_key])

In [24]:
chunked_data['X_High_30'].shape

(6285, 30)

In [28]:
trained_models = {}

for X, y in (chunk_pairs):
    X_data = chunked_data[X]
    y_data = chunked_data[y]
    print(X_data, y_data)

[[ 1592.9   1641.95  1635.5  ...  1713.7   1771.65  1795.45]
 [ 1641.95  1635.5   1639.   ...  1771.65  1795.45  1744.5 ]
 [ 1635.5   1639.    1628.25 ...  1795.45  1744.5   1742.8 ]
 ...
 [23214.7  22254.   22697.2  ... 25062.95 25010.35 24946.2 ]
 [22254.   22697.2  22468.7  ... 25010.35 24946.2  24737.5 ]
 [22697.2  22468.7  22923.9  ... 24946.2  24737.5  24909.05]] [ 1744.5   1742.8   1753.1  ... 24737.5  24909.05 25079.2 ]
[[ 1592.9   1641.95  1635.5  ...  1593.3   1575.85  1609.4 ]
 [ 1641.95  1635.5   1639.   ...  1575.85  1609.4   1557.85]
 [ 1635.5   1639.    1628.25 ...  1609.4   1557.85  1545.55]
 ...
 [22992.5  23049.95 22923.85 ... 25062.95 25010.35 24946.2 ]
 [23049.95 22923.85 22921.   ... 25010.35 24946.2  24737.5 ]
 [22923.85 22921.   22668.05 ... 24946.2  24737.5  24909.05]] [ 1557.85  1545.55  1555.5  ... 24737.5  24909.05 25079.2 ]
[[ 1592.9   1641.95  1635.5  ...  1359.1   1326.3   1311.7 ]
 [ 1641.95  1635.5   1639.   ...  1326.3   1311.7   1310.45]
 [ 1635.5   16

### 4. Define Neural Network Models

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, GRU, Bidirectional


def build_rnn(input_shape):
    model = Sequential([
        SimpleRNN(50, activation='tanh', input_shape=input_shape),
        Dense(1)   # regression output
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def build_lstm(input_shape):
    model = Sequential([
        LSTM(50, activation='tanh', input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def build_gru(input_shape):
    model = Sequential([
        GRU(50, activation='tanh', input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def build_bilstm(input_shape):
    model = Sequential([
        Bidirectional(LSTM(50, activation='tanh'), input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

In [26]:
ml_model = [("KNN", KNeighborsRegressor())]


dl_models = {
    "RNN": build_rnn,
    "LSTM": build_lstm,
    "GRU": build_gru,
    "Bidirectional_LSTM": build_bilstm
}

### 5. Training the Model

In [29]:
trained_models = {}

for X, y in tqdm(chunk_pairs):
    X_data = chunked_data[X]
    y_data = chunked_data[y]

    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y_data, test_size=0.1, random_state=42
    )

    # ML models
    for model_name, model in tqdm(ml_model):
        key = model_name + '_' + X[2:]
        model_copy = deepcopy(model)
        model_copy.fit(X_train, y_train)

        y_train_pred = model_copy.predict(X_train)
        y_test_pred = model_copy.predict(X_test)

        trained_models[key] = {
            'model': model_copy,
            'train_mae': mean_absolute_error(y_train, y_train_pred),
            'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
            'test_mae': mean_absolute_error(y_test, y_test_pred),
            'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred))
        }

    # DL models
    X_train_rnn = np.expand_dims(X_train, -1)
    X_test_rnn = np.expand_dims(X_test, -1)

    for model_name, builder in tqdm(dl_models.items()):
        key = model_name + '_' + X[2:]
        model_dl = builder((X_train.shape[1], 1))

        model_dl.fit(X_train_rnn, y_train, epochs=50, batch_size=8, verbose=0)

        y_train_pred = model_dl.predict(X_train_rnn).flatten()
        y_test_pred = model_dl.predict(X_test_rnn).flatten()

        trained_models[key] = {
            'model': model_dl,
            'train_mae': mean_absolute_error(y_train, y_train_pred),
            'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
            'test_mae': mean_absolute_error(y_test, y_test_pred),
            'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred))
        }


100%|██████████| 1/1 [00:06<00:00,  6.34s/it]


177/177 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


177/177 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


177/177 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


177/177 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


...

176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


176/176 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


176/176 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


176/176 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


100%|██████████| 3/3 [2:21:55<00:00, 2838.49s/it]


### 6. Model Statistics

In [30]:
results_df = pd.DataFrame([
    {"Model": name, **metrics}
    for name, metrics in trained_models.items()])

results_df.sort_values(by = 'test_mae', ascending = True)

,Model,model,train_mae,train_rmse,test_mae,test_rmse
10,KNN_High_90,KNeighborsRegressor(),36.929140,58.769966,48.790144,74.204848
5,KNN_High_60,KNeighborsRegressor(),36.850899,58.333589,52.819089,82.674612
0,KNN_High_30,KNeighborsRegressor(),42.646429,68.669169,56.933037,89.679421
1,RNN_High_30,"<Sequential name=sequential, built=True>",6386.214642,8840.065628,5889.220259,8443.691486
6,RNN_High_60,"<Sequential name=sequential_4, built=True>",6419.272066,8878.256804,5963.836583,8380.827956
11,RNN_High_90,"<Sequential name=sequential_8, built=True>",6434.589589,8882.542999,6157.088872,8599.560408
4,Bidirectional_LSTM_High_30,"<Sequential name=sequential_3, built=True>",6864.279826,9293.892113,6346.067570,8881.021585
9,Bidirectional_LSTM_High_60,"<Sequential name=sequential_7, built=True>",6847.724789,9288.696827,6367.249068,8783.309526
8,GRU_High_60,"<Sequential name=sequential_6, built=True>",7023.737292,9429.018835,6542.382753,8921.351181
3,GRU_High_30,"<Sequential name=sequential_2, built=True>",7062.769825,9446.612370,6543.584412,9028.903666


In [43]:
results_df['Model']

10                   KNN_High_90
5                    KNN_High_60
0                    KNN_High_30
1                    RNN_High_30
6                    RNN_High_60
11                   RNN_High_90
4     Bidirectional_LSTM_High_30
9     Bidirectional_LSTM_High_60
8                    GRU_High_60
3                    GRU_High_30
14    Bidirectional_LSTM_High_90
7                   LSTM_High_60
13                   GRU_High_90
2                   LSTM_High_30
12                  LSTM_High_90
Name: Model, dtype: str

In [49]:
mae_sorted_df = results_df.sort_values(by='test_mae', ascending=True)
model_types = pd.Series([i.split('_')[0] for i in mae_sorted_df['Model']])

model_counts = model_types.value_counts().sort_values(ascending=False)
model_counts.index


Index(['KNN', 'RNN', 'Bidirectional', 'GRU', 'LSTM'], dtype='str')

In [42]:
# Compare top 10 models by each metric
top_mae = results_df.nsmallest(10, 'test_mae')[['Model', 'test_mae']]
top_rmse = results_df.nsmallest(10, 'test_rmse')[['Model', 'test_rmse']]

print("Top 10 by MAE:\n", top_mae)
print("\nTop 10 by RMSE:\n", top_rmse)

Top 10 by MAE:
                          Model     test_mae
10                 KNN_High_90    48.790144
5                  KNN_High_60    52.819089
0                  KNN_High_30    56.933037
1                  RNN_High_30  5889.220259
6                  RNN_High_60  5963.836583
11                 RNN_High_90  6157.088872
4   Bidirectional_LSTM_High_30  6346.067570
9   Bidirectional_LSTM_High_60  6367.249068
8                  GRU_High_60  6542.382753
3                  GRU_High_30  6543.584412

Top 10 by RMSE:
                          Model    test_rmse
10                 KNN_High_90    74.204848
5                  KNN_High_60    82.674612
0                  KNN_High_30    89.679421
6                  RNN_High_60  8380.827956
1                  RNN_High_30  8443.691486
11                 RNN_High_90  8599.560408
9   Bidirectional_LSTM_High_60  8783.309526
4   Bidirectional_LSTM_High_30  8881.021585
8                  GRU_High_60  8921.351181
3                  GRU_High_30  9028.90366